## 1. Load and Clean Data

In [ ]:

import pandas as pd
from scipy import stats

# Load dataset
df = pd.read_csv("Customer_Churn_Template.csv")

# Outlier Removal using Z-score
z_scores = stats.zscore(df[['Total_Spend', 'Marketing_Spend']])
df = df[(abs(z_scores) < 3).all(axis=1)]

# Encode 'Churned' column
df['Churned'] = df['Churned'].map({'Yes': 1, 'No': 0})

df.head()


## 2. Logistic Regression - Churn Prediction

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report

X = df[['Total_Spend', 'Purchase_Frequency', 'Marketing_Spend', 'Seasonality_Index']]
y = df['Churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Hyperparameter tuning
param_grid = {'C': [0.01, 0.1, 1, 10]}
grid = GridSearchCV(LogisticRegression(), param_grid, cv=5)
grid.fit(X_train, y_train)

y_pred = grid.predict(X_test)
print(classification_report(y_test, y_pred))


## 3. Linear Regression - Predict Spend

In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X = df[['Marketing_Spend', 'Seasonality_Index']]
y = df['Total_Spend']

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

print("R^2 Score:", r2_score(y, y_pred))
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)


## 4. Statistical Testing (ANOVA and T-Test)

In [ ]:

import scipy.stats as stats

# ANOVA - Total Spend by Region
anova_groups = [group['Total_Spend'] for name, group in df.groupby('Region')]
f_stat, p_val = stats.f_oneway(*anova_groups)
print(f"ANOVA F-statistic: {f_stat}, p-value: {p_val}")

# T-Test - Marketing Spend for Churned vs. Not
churned = df[df['Churned'] == 1]['Marketing_Spend']
not_churned = df[df['Churned'] == 0]['Marketing_Spend']
t_stat, p_val = stats.ttest_ind(churned, not_churned)
print(f"T-Test statistic: {t_stat}, p-value: {p_val}")


## 5. K-Means Clustering and Segmentation

In [ ]:

from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

kmeans = KMeans(n_clusters=3, random_state=42)
df['Customer_Segment'] = kmeans.fit_predict(df[['Total_Spend', 'Purchase_Frequency']])

# Visualize Clusters
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='Total_Spend', y='Purchase_Frequency', hue='Customer_Segment', palette='Set1')
plt.title('Customer Segmentation')
plt.show()
